### TSC GRPO 训练 - new_prompt_training JSON 协议（保留仿真回溯评估）



In [ ]:
import os
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"



In [ ]:
import subprocess
result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value



### 加载模型



In [ ]:
from unsloth import FastLanguageModel
import torch
import os

max_seq_length = 2048  # 增加序列长度以容纳完整的 prompt + completion
lora_rank = 32

os.environ["HF_HOME"] = 'model'
os.environ["MODELSCOPE_CACHE"] = 'model'

# ====== 断点续训：有 checkpoint 就继续微调；没有就从 base 模型开始 ======
BASE_MODEL_DIR = "model/models/qwen3-4B-SFT"
CHECKPOINT_DIR = "checkpoints/newprompt_grpo_latest"

def _looks_like_checkpoint(path: str) -> bool:
    if not os.path.isdir(path):
        return False
    # PEFT/LoRA 常见标志文件（任意一个即可认为可加载）
    marker_files = [
        "adapter_config.json",
        "adapter_model.safetensors",
        "adapter_model.bin",
        "config.json",
    ]
    return any(os.path.isfile(os.path.join(path, f)) for f in marker_files)

resume_from = CHECKPOINT_DIR if _looks_like_checkpoint(CHECKPOINT_DIR) else BASE_MODEL_DIR
if resume_from == CHECKPOINT_DIR:
    print(f"✓ 检测到 checkpoint，将从此继续微调: {CHECKPOINT_DIR}")
else:
    print(f"ℹ 未检测到 checkpoint，将从基础模型开始: {BASE_MODEL_DIR}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = resume_from,
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    fast_inference = False,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.8,
)

# 若是从 base 模型开始，需要创建 LoRA；若是从 checkpoint 加载，一般已包含 LoRA（无需重复包一层）
if resume_from == BASE_MODEL_DIR:
    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank,
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha = lora_rank * 2,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
else:
    # 尽量开启梯度检查点（如果当前模型类型支持）
    try:
        model.gradient_checkpointing_enable()
    except Exception:
        pass

    # 保险：有些版本从 checkpoint 加载后 LoRA 参数可能默认不可训练
    _trainable = [p for p in model.parameters() if p.requires_grad]
    if len(_trainable) == 0:
        for name, p in model.named_parameters():
            if "lora" in name.lower():
                p.requires_grad = True
        print("⚠️ checkpoint 加载后未检测到可训练参数，已强制启用 LoRA 参数训练")



### 场景发现 & 仿真器



In [ ]:
import json
import random
import sys
import xml.etree.ElementTree as ET
from collections import deque
from typing import Dict, List, Tuple


def discover_environments(environments_root: str) -> Dict[str, Dict]:
    environments: Dict[str, Dict] = {}
    if not os.path.isdir(environments_root):
        print(f"警告: environments 目录不存在: {environments_root}")
        return environments

    for scenario_name in sorted(os.listdir(environments_root)):
        scenario_dir = os.path.join(environments_root, scenario_name)
        if not os.path.isdir(scenario_dir) or scenario_name.startswith('.'):
            continue

        sumocfg = None
        for f in os.listdir(scenario_dir):
            if f.endswith('.sumocfg'):
                sumocfg = os.path.join(scenario_dir, f)
                break

        net_xml = None
        for f in os.listdir(scenario_dir):
            if f.endswith('.net.xml'):
                net_xml = os.path.join(scenario_dir, f)
                break

        if sumocfg and net_xml:
            tl_ids = extract_traffic_light_ids(net_xml)
            if tl_ids:
                environments[scenario_name] = {
                    'sumocfg': sumocfg,
                    'net': net_xml,
                    'tl_ids': tl_ids,
                }

    return environments


def extract_traffic_light_ids(net_xml_path: str) -> List[str]:
    tl_ids: List[str] = []
    try:
        for _event, elem in ET.iterparse(net_xml_path, events=("end",)):
            if elem.tag == "tlLogic":
                tl_id = elem.attrib.get("id")
                if tl_id and tl_id not in tl_ids:
                    tl_ids.append(tl_id)
                elem.clear()
    except Exception as e:
        print(f"解析 {net_xml_path} 失败: {e}")
    return tl_ids


ENVIRONMENTS_ROOT = os.path.join(os.getcwd(), 'sumo_simulation/environments')
AVAILABLE_ENVIRONMENTS = discover_environments(ENVIRONMENTS_ROOT)
print(f"发现场景数: {len(AVAILABLE_ENVIRONMENTS)}")
total_intersections = sum(len(env['tl_ids']) for env in AVAILABLE_ENVIRONMENTS.values())
print(f"总交叉口数量: {total_intersections}")

# 添加 sumo_simulation 目录到 Python 路径
sumo_sim_path = os.path.join(os.getcwd(), 'sumo_simulation')
if sumo_sim_path not in sys.path:
    sys.path.insert(0, sumo_sim_path)

from sumo_simulator import SUMOSimulator



### new_prompt 构造、解析、奖励（含回溯评估）



In [ ]:
import datetime
import gc

from scu_tsc_newprompt.phase_parser import get_net_phase_minmax_one_based, get_phase_order_one_based
from scu_tsc_newprompt.constraint_sampler import sample_phase_limits_hybrid
from scu_tsc_newprompt.prompt_builder import build_cycle_predict_input_json, wrap_prompt_with_markers
from scu_tsc_newprompt.rewards import (
    score_constraints_and_format,
    AdaptiveScaler,
    compute_sim_reward_adaptive,
    compute_total_reward,
)


SYSTEM_PROMPT = """你是交通信号配时优化专家。
你将收到一个 JSON（用【cycle_predict_input_json】...【/cycle_predict_input_json】包裹）。
你的任务是：在满足硬约束前提下，输出下一周期各相位最终绿灯时间 final（单位：秒）。
注意：history 数据可能不完整（例如 only recent_cycles；yesterday_same_time/last_week_same_time 可能为 null），必须基于可用部分输出结果。
只输出最终 JSON 数组(list)，不要输出任何解释/过程。
"""

USER_INSTRUCTIONS = """任务（必须完成）：
1) 基于输入 JSON 的 history.* 历史数据，自行决定预测算法/模型/参数，预测“下一周期各相位需求强度”（仅在内部使用，不输出过程/中间值）。
2) 在满足硬约束前提下，输出下一周期各相位最终绿灯时间 final（单位：秒）。

要求（必须遵守）：
1) history 数据可能不完整（仅 recent_cycles；windows.yesterday_same_time / last_week_same_time 可能为 null）；这是正常情况，请基于可用部分完成预测，并且必须继续输出结果。
2) 若 history 中存在有效数据，请使用它完成预测并体现在结果中；不要忽略 history 数据随意分配。
3) 若 history 数据缺失/异常/几乎全为 0，仍需输出满足硬约束的可执行方案，但不得编造不存在的数据。
4) 只输出最终 JSON，不要输出任何解释/过程。

输出要求（必须严格遵守）：
1) 只输出最终 JSON（不要任何说明、不要 Markdown）。
2) JSON 顶层必须是数组(list)；数组长度必须等于相位数。
3) 数组元素必须为对象：{"phase_id": <int>, "final": <int>}；不允许输出其它字段。
4) phase_id 必须覆盖全部相位且不重复，并且顺序必须与 phase_order 完全一致。
"""


def build_user_prompt(payload: dict) -> str:
    return wrap_prompt_with_markers(payload) + "\n\n" + USER_INSTRUCTIONS


def collect_phase_waits_snapshot(simulator: SUMOSimulator, tl_id: str, phase_order: List[int]) -> List[dict]:
    """用当前时刻各相位 incoming lanes 的停止车辆数，作为 avg_wait 的代理。"""
    import traci

    waits = []
    for phase_id in phase_order:
        phase_idx = phase_id - 1
        lanes = simulator.get_phase_controlled_lanes(tl_id, phase_idx).get('incoming_lanes', [])
        if not lanes:
            avg = 0.0
        else:
            total = 0.0
            for ln in lanes:
                try:
                    total += traci.lane.getLastStepHaltingNumber(ln)
                except Exception:
                    pass
            avg = total / max(1, len(lanes))
        waits.append({'phase_id': int(phase_id), 'avg_wait': float(round(avg, 2))})
    return waits


def evaluate_plan_once(simulator: SUMOSimulator, tl_id: str, plan: List[dict]) -> dict:
    """执行一个整周期方案并返回指标（在当前仿真状态下直接前进）。"""
    import traci

    # 统一收集所有相位可能涉及的 incoming lanes（用于粗略 passed/queue 统计）
    phase_info = simulator.get_phase_info(tl_id)
    n = int(phase_info.get('num_phases', 0))
    all_lanes = set()
    for idx in range(n):
        all_lanes.update(simulator.get_phase_controlled_lanes(tl_id, idx).get('incoming_lanes', []))
    all_lanes = list(all_lanes)

    vehicles_before = set()
    for ln in all_lanes:
        try:
            vehicles_before.update(traci.lane.getLastStepVehicleIDs(ln))
        except Exception:
            pass

    total_queue_proxy = 0.0
    for step in plan:
        pid = int(step['phase_id'])
        dur = int(step['final'])
        traci.trafficlight.setPhase(tl_id, pid - 1)
        for _ in range(max(0, dur)):
            traci.simulationStep()
            q = 0.0
            for ln in all_lanes:
                try:
                    q += traci.lane.getLastStepHaltingNumber(ln)
                except Exception:
                    pass
            total_queue_proxy += q

    vehicles_after = set()
    queue_end = 0.0
    for ln in all_lanes:
        try:
            vehicles_after.update(traci.lane.getLastStepVehicleIDs(ln))
            queue_end += traci.lane.getLastStepHaltingNumber(ln)
        except Exception:
            pass

    passed = len(vehicles_before - vehicles_after)
    return {
        'passed_vehicles': float(passed),
        'queue_vehicles': float(queue_end),
        'total_queue_proxy': float(total_queue_proxy),
        'sim_time': float(traci.simulation.getTime()),
    }


def evaluate_multiple_plans_with_rollback(simulator: SUMOSimulator, tl_id: str, plans: List[List[dict]]) -> List[dict]:
    """对多个候选方案做回溯评估：同一检查点下逐个执行/恢复，最后回到原状态。"""
    state_file = simulator.save_simulation_state(tl_id)
    results = []
    try:
        for p in plans:
            simulator.restore_simulation_state(state_file)
            results.append(evaluate_plan_once(simulator, tl_id, p))
        simulator.restore_simulation_state(state_file)
    finally:
        simulator.cleanup_state_file(state_file)
    return results



### GRPO 训练（每步：构造 JSON prompt → 生成 N 个方案 → 回溯评估 → 更新 → 执行最优推进）



In [ ]:
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
import re
import sys

TRAIN_CONFIG = {
    'gui': False,
    'max_tl_per_scenario': 4,
    'warmup_steps': 80,
    'steps_per_tl': 10,
    'num_generations': 21,
    'learning_rate': 2e-6,
    'max_new_tokens': 256,
    'temperature': 0.8,
    'top_p': 0.95,
    'top_k': 50,
    'gradient_accumulation_steps': 4,
    'log_interval': 2,
    'clear_cache_interval': 5,
    # recent_cycles 窗口
    'recent_cycles_maxlen': 12,
    # ========== 新版 reward 配置 ==========
    # 仿真指标权重（归一化后加权）
    'w_passed': 1.0,      # 通过车辆数权重
    'w_queue': 1.0,       # 排队车辆数权重（负向）
    'w_proxy': 0.2,       # 累积排队权重（负向，辅助）
    # 总 reward 组合权重
    'w_sim': 1.0,         # 仿真 reward 权重
    'w_constraint': 0.3,  # 约束惩罚权重
    # 软约束归一化尺度（秒）
    'D0': 30.0,
    # 日志文件路径
    'log_file': 'training.log',
    # 跳过的交叉口 ID 列表（在所有场景中跳过这些 tl_id）
    'skip_tl_ids': ['nt12'],  # 例如: ['nt12', 'nt13', 'junction_abc']
}


# ========== 日志和训练跳过工具 ==========
class TeeLogger:
    """同时输出到控制台和文件的日志类"""
    def __init__(self, log_file):
        self.log_file = log_file
        self.terminal = sys.stdout
        # 以追加模式打开日志文件
        self.log = open(log_file, 'a', encoding='utf-8', buffering=1)
    
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
    
    def flush(self):
        self.terminal.flush()
        self.log.flush()
    
    def close(self):
        self.log.close()


def parse_completed_trainings(log_file: str) -> set:
    """
    解析 training.log，返回已完成训练的 (scenario, tl_id) 组合集合。
    
    判断标准：
    1. 找到 "=== [数字/数字] scenario/tl_id ===" 开始一个训练记录
    2. 在该记录后找到 "✓ checkpoint 已保存到" 确认训练完成
    """
    if not os.path.exists(log_file):
        return set()
    
    completed = set()
    current_pair = None
    
    with open(log_file, 'r', encoding='utf-8') as f:
        for line in f:
            # 匹配训练开始行：=== [数字/数字] scenario/tl_id ===
            match_start = re.search(r'===\s*\[\d+/\d+\]\s+([^/]+)/(\S+)\s+===', line)
            if match_start:
                current_pair = (match_start.group(1), match_start.group(2))
                continue
            
            # 匹配 checkpoint 保存行：✓ checkpoint 已保存到
            if current_pair and '✓ checkpoint 已保存到' in line:
                completed.add(current_pair)
                current_pair = None
    
    return completed


def compute_grpo_loss(model, tokenizer, prompt_ids, completion_ids, rewards):
    device = next(model.parameters()).device
    
    # 截断 prompt 和 completion 以适应 max_seq_length
    max_len = max_seq_length
    prompt_len = prompt_ids.shape[1]
    completion_len = completion_ids.shape[1]
    total_len = prompt_len + completion_len
    
    if total_len > max_len:
        # 优先保留 completion，截断 prompt
        available_for_prompt = max(256, max_len - completion_len)  # 至少保留 256 个 prompt token
        if prompt_len > available_for_prompt:
            prompt_ids = prompt_ids[:, -available_for_prompt:]  # 保留 prompt 末尾
            prompt_len = available_for_prompt
        # 如果还是超长，截断 completion
        if prompt_len + completion_len > max_len:
            completion_ids = completion_ids[:, :max_len - prompt_len]
            completion_len = completion_ids.shape[1]
    
    input_ids = torch.cat([prompt_ids, completion_ids], dim=1).to(device)
    outputs = model(input_ids=input_ids)
    logits = outputs.logits

    # 确保维度匹配
    completion_logits = logits[:, prompt_len - 1 : prompt_len - 1 + completion_len, :]
    completion_targets = completion_ids.to(device)
    
    # 确保维度一致
    min_len = min(completion_logits.shape[1], completion_targets.shape[1])
    completion_logits = completion_logits[:, :min_len, :]
    completion_targets = completion_targets[:, :min_len]

    log_probs = F.log_softmax(completion_logits, dim=-1)
    token_log_probs = torch.gather(log_probs, 2, completion_targets.unsqueeze(-1)).squeeze(-1)

    attention_mask = (completion_targets != tokenizer.pad_token_id).float()
    seq_log_probs = (token_log_probs * attention_mask).sum(dim=1) / (attention_mask.sum(dim=1) + 1e-8)

    rewards_tensor = torch.tensor(rewards, dtype=torch.float32, device=device)
    
    # 防止 std=0 的情况（发生在所有 reward 相同时）
    if rewards_tensor.std() < 1e-6:
        # 所有 reward 相同时，给一个极小的随机扰动，避免 loss=0
        rewards_tensor = rewards_tensor + torch.randn_like(rewards_tensor) * 1e-4
    
    rewards_normalized = (rewards_tensor - rewards_tensor.mean()) / (rewards_tensor.std() + 1e-8)

    policy_loss = -(seq_log_probs * rewards_normalized).mean()

    return policy_loss, {
        'policy_loss': float(policy_loss.item()),
        'mean_reward': float(rewards_tensor.mean().item()),
        'reward_std': float(rewards_tensor.std().item()),
    }


def build_scenario_tl_list() -> List[Tuple[str, str]]:
    pairs: List[Tuple[str, str]] = []
    for scenario_name, info in AVAILABLE_ENVIRONMENTS.items():
        for tl_id in info['tl_ids'][: TRAIN_CONFIG['max_tl_per_scenario']]:
            pairs.append((scenario_name, tl_id))
    random.shuffle(pairs)
    return pairs


def train_one_tl(scenario_name: str, tl_id: str) -> bool:
    env_info = AVAILABLE_ENVIRONMENTS[scenario_name]

    simulator = SUMOSimulator(
        config_file=env_info['sumocfg'],
        junctions_file=None,
        gui=TRAIN_CONFIG['gui'],
    )
    if not simulator.start_simulation():
        print(f"✗ 启动失败: {scenario_name}/{tl_id}")
        return False

    # warmup
    for _ in range(TRAIN_CONFIG['warmup_steps']):
        if simulator.is_connected():
            simulator.step()

    # phase order from net.xml (1-based)
    phase_order = get_phase_order_one_based(env_info['net'], tl_id)
    if not phase_order:
        print(f"✗ 未解析到相位: {scenario_name}/{tl_id}")
        simulator.close()
        return False

    net_minmax = get_net_phase_minmax_one_based(env_info['net'], tl_id)

    # history buffer: recent_cycles only
    recent_cycles_buf = deque(maxlen=TRAIN_CONFIG['recent_cycles_maxlen'])

    # 自适应尺度器（按路口维护 P95）
    scaler = AdaptiveScaler()
    
    # baseline：上一周期 best_plan 的仿真结果（用于相对改进）
    baseline_result: Dict[str, float] | None = None
    baseline_sim_reward = -float('inf')  # 追踪baseline的sim_reward，用于有条件更新

    # optimizer
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=TRAIN_CONFIG['learning_rate'], weight_decay=0.1)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(TRAIN_CONFIG['steps_per_tl'] * 0.1),
        num_training_steps=TRAIN_CONFIG['steps_per_tl'],
    )
    optimizer.zero_grad()

    for step_idx in range(TRAIN_CONFIG['steps_per_tl']):
        if not simulator.is_connected():
            break

        # 随机化约束（每步变化），同一 (scenario, tl, step) 可复现
        rng = random.Random((hash(scenario_name) ^ hash(tl_id) ^ step_idx) & 0xFFFFFFFF)
        phase_limits = sample_phase_limits_hybrid(phase_order, net_minmax, rng=rng)

        payload = build_cycle_predict_input_json(
            scenario_name=scenario_name,
            tl_id=tl_id,
            phase_order=phase_order,
            phase_limits=phase_limits,
            recent_cycles=list(recent_cycles_buf),
            include_windows_recent_past=True,
            windows_recent_past=None,
        )

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(payload)},
        ]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids

        device = next(model.parameters()).device
        prompt_ids_batch = prompt_ids.repeat(TRAIN_CONFIG['num_generations'], 1).to(device)

        # generate
        FastLanguageModel.for_inference(model)
        with torch.no_grad():
            generated = model.generate(
                input_ids=prompt_ids_batch,
                max_new_tokens=TRAIN_CONFIG['max_new_tokens'],
                temperature=TRAIN_CONFIG['temperature'],
                top_p=TRAIN_CONFIG['top_p'],
                top_k=TRAIN_CONFIG['top_k'],
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        completion_ids = generated[:, prompt_ids.shape[1] :].clone().detach()
        FastLanguageModel.for_training(model)
        completions = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)

        # constraint score + parse（软约束检查，只检查 min_green/max_green）
        # 新版：约束分数在 [-1.5, 0] 范围（满足为 0，越违反越负）
        parsed_plans: List[List[dict] | None] = []
        constraint_scores: List[float] = []
        for i, txt in enumerate(completions):
            s, _info, plan = score_constraints_and_format(
                completion_text=txt,
                phase_order=phase_order,
                phase_limits=phase_limits,
                D0=TRAIN_CONFIG['D0'],
            )
            constraint_scores.append(float(s))
            parsed_plans.append(plan)
            
            # === 调试：每2 steps 打印所有 completion ===
            if step_idx % 2 == 0:
                print(f"\n[DEBUG step={step_idx} gen={i}] constraint_score={s:.2f} error={_info.get('error')}")
                print(f"  bounds_deviation_sec={_info.get('bounds_deviation_sec', 0):.1f}")
                print(f"  completion={repr(txt[:300])}")

        # 回溯仿真评估：只对“满足全部硬约束”的候选做回溯（省算力）
        plans_for_sim: List[List[dict]] = []
        sim_map: Dict[int, int] = {}
        for i, p in enumerate(parsed_plans):
            if p is not None:  # 只要 plan 解析成功就进行仿真评估（软约束）
                sim_map[i] = len(plans_for_sim)
                plans_for_sim.append(p)

        # 仿真评估：计算交通指标（使用自适应尺度 + baseline 相对改进）
        sim_rewards = [0.0] * len(completions)
        sim_results_all: List[Dict[str, float] | None] = [None] * len(completions)
        
        if plans_for_sim:
            sim_results = evaluate_multiple_plans_with_rollback(simulator, tl_id, plans_for_sim)
            
            # 更新自适应尺度器（用本轮评估结果）
            scaler.add_observations_batch(sim_results)
            
            for i in sim_map:
                r = sim_results[sim_map[i]]
                sim_results_all[i] = r
                
                # 使用自适应尺度 + baseline 相对改进计算 sim_reward
                sim_reward, _sim_info = compute_sim_reward_adaptive(
                    result=r,
                    scaler=scaler,
                    baseline_result=baseline_result,
                    w_passed=TRAIN_CONFIG['w_passed'],
                    w_queue=TRAIN_CONFIG['w_queue'],
                    w_proxy=TRAIN_CONFIG['w_proxy'],
                )
                sim_rewards[i] = float(sim_reward)

        # total reward = w_sim * sim_reward + w_constraint * constraint_score
        rewards = [
            compute_total_reward(
                constraint_score=constraint_scores[i],
                sim_reward=sim_rewards[i],
                w_sim=TRAIN_CONFIG['w_sim'],
                w_constraint=TRAIN_CONFIG['w_constraint'],
            )
            for i in range(len(completions))
        ]

        # GRPO update
        prompt_ids_for_loss = prompt_ids_batch.clone()
        completion_ids_for_loss = completion_ids.clone()
        loss, info = compute_grpo_loss(model, tokenizer, prompt_ids_for_loss, completion_ids_for_loss, rewards)
        (loss / TRAIN_CONFIG['gradient_accumulation_steps']).backward()

        if (step_idx + 1) % TRAIN_CONFIG['gradient_accumulation_steps'] == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        # 执行最优（推进单条真实轨迹）
        best_i = max(range(len(rewards)), key=lambda i: rewards[i])
        best_plan = parsed_plans[best_i]

        if best_plan is None:
            for _ in range(10):
                if simulator.is_connected():
                    simulator.step()
            # best_plan 无效时不更新 baseline
        else:
            waits = collect_phase_waits_snapshot(simulator, tl_id, phase_order)
            recent_cycles_buf.append(
                {
                    'time': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'phase_waits': waits,
                }
            )
            # 执行最优 plan 并获取实际仿真结果作为下一轮的 baseline
            best_result = evaluate_plan_once(simulator, tl_id, best_plan)
            
            # 有条件更新 baseline：只在 sim_reward 改善时更新
            current_sim_reward = sim_rewards[best_i]
            if current_sim_reward > baseline_sim_reward:
                baseline_result = best_result
                baseline_sim_reward = current_sim_reward

        if (step_idx + 1) % TRAIN_CONFIG['log_interval'] == 0:
            valid_plans = sum(1 for p in parsed_plans if p is not None)
            scales = scaler.get_scales()
            print(
                f"[{scenario_name}/{tl_id}] step {step_idx+1}/{TRAIN_CONFIG['steps_per_tl']} "
                f"loss={info['policy_loss']:.4f} mean_r={info['mean_reward']:.3f} std={info['reward_std']:.3f} "
                f"valid={valid_plans}/{len(parsed_plans)} "
                f"best_c={constraint_scores[best_i]:.2f} best_sim={sim_rewards[best_i]:.2f} "
                f"scales=({scales[0]:.1f},{scales[1]:.1f},{scales[2]:.0f})"
            )

        # cleanup
        del loss, generated, completion_ids_for_loss, prompt_ids_for_loss
        gc.collect()
        if torch.cuda.is_available() and (step_idx + 1) % TRAIN_CONFIG['clear_cache_interval'] == 0:
            torch.cuda.empty_cache()

    simulator.close()
    return True


pairs = build_scenario_tl_list()
print(f"训练组合数: {len(pairs)}")

# 先跑前若干个路口（避免一次训练过大）
MAX_TRAIN_TLS = 5000
import shutil
save_dir = "checkpoints/newprompt_grpo_latest"

# ========== 设置日志输出 ==========
log_file = TRAIN_CONFIG['log_file']
# 确保日志文件存在（不存在则创建空白文件）
if not os.path.exists(log_file):
    open(log_file, 'w', encoding='utf-8').close()
    print(f"✓ 创建新日志文件: {log_file}")
else:
    print(f"✓ 使用现有日志文件: {log_file}")

# 重定向 stdout 到 TeeLogger（同时输出到控制台和文件）
tee_logger = TeeLogger(log_file)
sys.stdout = tee_logger

# ========== 解析已完成的训练组合 ==========
completed_trainings = parse_completed_trainings(log_file)
print(f"已完成训练的组合数: {len(completed_trainings)}")
if completed_trainings:
    print(f"已完成的前几个: {list(completed_trainings)[:5]}")

# ========== 跳过列表配置 ==========
skip_tl_ids = set(TRAIN_CONFIG['skip_tl_ids'])
if skip_tl_ids:
    print(f"配置跳过的交叉口: {list(skip_tl_ids)}")

# ========== 主训练循环 ==========
skipped_count = 0
skipped_by_config_count = 0
trained_count = 0

for i, (sc, tl) in enumerate(pairs[:MAX_TRAIN_TLS], start=1):
    # 检查是否在配置的跳过列表中
    if tl in skip_tl_ids:
        print(f"\n⊘ 跳过 [{i}/{min(MAX_TRAIN_TLS, len(pairs))}] {sc}/{tl} (配置跳过)")
        skipped_by_config_count += 1
        continue
    
    # 检查是否已经训练过
    if (sc, tl) in completed_trainings:
        print(f"\n⊘ 跳过 [{i}/{min(MAX_TRAIN_TLS, len(pairs))}] {sc}/{tl} (已训练)")
        skipped_count += 1
        continue
    
    print(f"\n=== [{i}/{min(MAX_TRAIN_TLS, len(pairs))}] {sc}/{tl} ===")
    _ok = train_one_tl(sc, tl)
    trained_count += 1
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # 每训练完一个交叉口保存一次 checkpoint（覆盖保存）
    if os.path.isdir(save_dir):
        shutil.rmtree(save_dir)
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)
    print(f"✓ checkpoint 已保存到 {save_dir} (完成 {trained_count}/{min(MAX_TRAIN_TLS, len(pairs))} 个交叉口)")

# ========== 训练结束，输出统计 ==========
print(f"\n{'='*60}")
print(f"训练完成！")
print(f"  本次训练: {trained_count} 个交叉口")
print(f"  跳过 (配置): {skipped_by_config_count} 个交叉口")
print(f"  跳过 (已训练): {skipped_count} 个交叉口")
print(f"  总计: {len(completed_trainings) + trained_count} 个交叉口已完成")
print(f"{'='*60}")

# 恢复 stdout 并关闭日志文件
sys.stdout = tee_logger.terminal
tee_logger.close()
print("✓ 日志已保存")

